# Use Case: Product Brainstorming with a Simulated Focus Group

This notebook demonstrates how TinyTroupe can be used to simulate a focus group for brainstorming new product ideas. We will create a small group of agents with different professional backgrounds and guide them through a multi-stage discussion to generate and refine feature ideas for Microsoft Word, enhanced with AI.

The simulation will cover:
1. Setting up a "Focus Group" world with diverse agents (a data scientist, an architect, and a physician).
2. An initial round where agents introduce themselves and discuss problems in their respective fields.
3. A second round where agents are tasked to brainstorm AI features for Microsoft Word that could address those problems.
4. A final round where one agent is assigned the role of rapporteur to consolidate the discussed ideas.
5. Using `ResultsExtractor` to parse and present the final list of brainstormed ideas.

## 1. Setup and Imports

Import necessary classes: `TinyPerson` for agents, `TinyWorld` for the environment, and example helpers to quickly create our agents with pre-defined personas (`create_lisa_the_data_scientist`, etc.). `sys` is used to adjust the Python path.

In [ ]:
import json
import sys
# If running from 'examples/use_cases/', this adds the parent directory of 'examples' (project root) to Python path.
sys.path.insert(0, '../..') 

import tinytroupe # Initializes configuration, logging, etc.
from tinytroupe.agent import TinyPerson
from tinytroupe.environment import TinyWorld
# Import pre-defined agent creation functions from the examples module
from tinytroupe.examples import create_lisa_the_data_scientist, create_oscar_the_architect, create_marcos_the_physician
from tinytroupe.extraction import ResultsExtractor # To parse the final output

## 2. Create the Focus Group Environment and Agents

We'll set up a `TinyWorld` named "Focus group" and populate it with three agents from different professions to ensure a variety of perspectives.

In [ ]:
# Create a world and add our pre-defined agents
world = TinyWorld("Focus group", 
                  agents=[
                      create_lisa_the_data_scientist(), 
                      create_oscar_the_architect(), 
                      create_marcos_the_physician()
                  ])
print(f"Created world: '{world.name}' with agents: {[agent.name for agent in world.agents]}")

## 3. Round 1: Introductions and Problem Statements

The facilitator (user) prompts the agents to introduce themselves and discuss the major problems and challenges in their work and industry. This sets the stage for identifying areas where new product features might help.

In [ ]:
# Broadcast the initial prompt to all agents in the world
world.broadcast("""
                Hello everyone! Let's start by introducing ourselves. What is your job and what are some major problems you face 
                in your work? What are major challenges for your industry as a whole? Don't discuss solutions yet, 
                just the problems you face.
                """)

# Run the simulation for one step to allow agents to respond to the introduction prompt
print("\n--- Round 1: Agent Introductions and Problem Statements ---")
world.run(1) 

## 4. Round 2: Brainstorming AI Features for Microsoft Word

The facilitator now directs the agents to brainstorm AI feature ideas for Microsoft Word, specifically targeting the problems they previously identified. They are encouraged to "think big" and not worry about implementation details at this stage.

In [ ]:
world.broadcast("""
                Ok, great. Now please add more details to these ideas - we need to understand them better. How would they actually integrate with Word? 
                Can you provide some concrete examples of what customers could do?
                """)

# Run the simulation for a few more steps to allow for discussion and idea generation
print("\n--- Round 2: Brainstorming AI Features for Word ---")
world.run(2) # Allowing 2 steps for more back-and-forth

## 5. Round 3: Consolidating Ideas

One agent, Lisa, is assigned the role of rapporteur to consolidate the brainstormed ideas into a structured list.

In [ ]:
# Get a reference to Lisa, who will act as the rapporteur
rapporteur = world.get_agent_by_name("Lisa Carter")

if rapporteur:
    print(f"\n--- Round 3: {rapporteur.name} Consolidating Ideas ---")
    rapporteur.listen_and_act("Can you please consolidate the ideas that the group came up with? Provide a lot of details on each idea, and complement anything missing.")
else:
    print("Error: Rapporteur agent 'Lisa Carter' not found.")

## 6. Extracting the Consolidated Ideas

Finally, we use `ResultsExtractor` to parse Lisa's (the rapporteur's) final response and extract the consolidated list of AI feature ideas for Microsoft Word, including details like benefits and drawbacks.

In [ ]:
if rapporteur:
    extractor = ResultsExtractor()
    
    # Define the objective for the extractor, matching the structure of the rapporteur's expected output
    extraction_objective=("Consolidate the ideas that the group came up with. "
                          "Each idea should be an item in a list. "
                          "For each idea, include its title, a detailed description, key_benefits (as a list of strings), "
                          "and drawbacks (as a list of strings), if any were mentioned.")

    # The situation helps the extractor understand the context of the agent's interaction history
    situation="A focus group brainstormed AI feature ideas for Microsoft Word. Lisa Carter was asked to consolidate these ideas."

    # Define the expected fields for the structured output
    fields=[
        {"name": "ideas", "type": "list", "description": "A list of all brainstormed ideas.",
         "element_schema": {
             "type": "dict",
             "keys": {
                 "title": "str",
                 "description": "str",
                 "key_benefits": {"type": "list", "element_schema": "str"},
                 "drawbacks": {"type": "list", "element_schema": "str"}
             }
         }}
    ]
    
    consolidated_ideas = extractor.extract_results_from_agent(
        person=rapporteur, 
        extraction_objective=extraction_objective, 
        situation=situation,
        fields=fields,
        verbose=False # Set to True to see LLM interaction during extraction
    )

    print("\n--- Extracted Brainstormed Ideas ---")
    if consolidated_ideas and consolidated_ideas.get('ideas'):
        for i, idea in enumerate(consolidated_ideas['ideas']):
            print(f"\nIdea {i+1}: {idea.get('title', 'N/A')}")
            print(f"  Description: {idea.get('description', 'N/A')}")
            print(f"  Benefits: {', '.join(idea.get('key_benefits', [])) if idea.get('key_benefits') else 'N/A'}")
            print(f"  Drawbacks: {', '.join(idea.get('drawbacks', [])) if idea.get('drawbacks') else 'N/A'}")
    else:
        print("Could not extract consolidated ideas.")
else:
    print("Skipping extraction as rapporteur agent was not found.")

This notebook illustrates a structured approach to using TinyTroupe for collaborative idea generation, simulating a multi-turn focus group discussion and leveraging specific agent capabilities for summarizing outcomes.